<a href="https://colab.research.google.com/github/IrumShehryar/ML-NLP-Coursework/blob/main/nlp/04-neural-network/Seq2Seq/Project01_TextGeneration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import the libraries

In [ ]:
# Reference : https://www.kaggle.com/code/annalee7/poetry-text-generation-lstm

In [1]:
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from keras.models import Model
from keras.layers import Dense, Embedding, Input, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.optimizers import Adam
import tensorflow as tf

# Define the parameters

In [2]:
vocab_size = 3000
embedding_size = 50
hidden_size = 25

# Read the data and create input and target sentences

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
%cd /content/drive/MyDrive/NLP-course/Course Material/Seq2Seq Network

/content/drive/MyDrive/NLP-course/Course Material/Seq2Seq Network


In [5]:
X = [] # input text
y = [] # target text
for line in open('MyText.txt'):
  line = line.rstrip()
  if not line:
    continue

  input_line = '<start> ' + line
  target_line = line + ' <end>'

  X.append(input_line)
  y.append(target_line)


total_lines = X + y

# Tokenization

In [6]:
tokenizer = Tokenizer(num_words = vocab_size, filters='') # Here we donot want to filter anything therefore filter = " "
                                                            # means that filter is empty string. This will ensure the angle
                                                            # signs of our tokens will retain.
tokenizer.fit_on_texts(total_lines)
input_sequences = tokenizer.texts_to_sequences(X)
target_sequences = tokenizer.texts_to_sequences(y)

# Get the sequence length

In [7]:
seq_len = max(len(s) for s in input_sequences)
print('Maximum seq length:', seq_len)

Maximum seq length: 12


# Word2index mapping

In [8]:
word2idx = tokenizer.word_index
print('Found %s unique tokens.' % len(word2idx))
assert('<start>' in word2idx)
assert('<end>' in word2idx)

Found 3056 unique tokens.


# Padding the sequence to get N x T

In [13]:
input_sequences = pad_sequences(input_sequences, maxlen = seq_len, padding='post')
target_sequences = pad_sequences(target_sequences, maxlen = seq_len, padding='post')
print('Data Shape:', input_sequences.shape)

Data Shape: (1436, 12)


# Create one hot of the targets as we cannot use cross entropy in keras because we have t targets for each input

In [14]:
one_hot_targets = np.zeros((len(input_sequences), seq_len, vocab_size))
for i, target_sequence in enumerate(target_sequences):
  for t, word in enumerate(target_sequence):
    if word > 0:
      one_hot_targets[i, t, word] = 1

# Create an LSTM Model

In [15]:
input_ = Input(shape=(seq_len, )) # input sequence
h_i = Input(shape=(hidden_size,))       # hidden state
c_i = Input(shape=(hidden_size,))       # cell state
# we pass initial states and cell states because we want to control them. we dont want keras to initialize them randomly
# because we want consistency.
embedding_layer = Embedding(vocab_size, embedding_size , input_length = seq_len)
x = embedding_layer(input_)
lstm = LSTM(hidden_size, return_sequences=True, return_state=True) # return_sequences=True because we need sequences
                                                                  # return_state=True, we need states later
x, _, _ = lstm(x, [h_i, c_i]) # only need x here
dense = Dense(vocab_size, activation='softmax')
output = dense(x)
model = Model([input_, h_i, c_i], output)

# Compile the model

In [16]:
model.compile( loss='categorical_crossentropy', optimizer=Adam(learning_rate = 0.01), metrics=['accuracy'])

# Here accuracy is uninterpretable because there are so many words that comes after the particular word.

# Train the model

In [17]:
hidden_state = np.zeros((len(input_sequences), hidden_size)) # Creating initial hidden state
cell_state = np.zeros((len(input_sequences), hidden_size))
hist = model.fit([input_sequences, hidden_state, cell_state], one_hot_targets,
  batch_size = 64,
  epochs = 20, # train for 3000 epochs
  validation_split = 0.2)


Epoch 1/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 5s 129ms/step - accuracy: 0.0783 - loss: 5.1389 - val_accuracy: 0.0833 - val_loss: 4.6389
Epoch 2/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 2s 110ms/step - accuracy: 0.0833 - loss: 4.3887 - val_accuracy: 0.0833 - val_loss: 4.7598
Epoch 3/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 2s 137ms/step - accuracy: 0.0833 - loss: 4.3178 - val_accuracy: 0.0833 - val_loss: 4.7668
Epoch 4/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 182ms/step - accuracy: 0.0833 - loss: 4.2641 - val_accuracy: 0.0833 - val_loss: 4.7201
Epoch 5/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 145ms/step - accuracy: 0.0833 - loss: 4.1945 - val_accuracy: 0.0833 - val_loss: 4.6916
Epoch 6/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 2s 99ms/step - accuracy: 0.0873 - loss: 4.1191 - val_accuracy: 0.0854 - val_loss: 4.6728
Epoch 7/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 2s 100ms/step - accuracy: 0.0926 - loss: 4.0428 - val_accuracy: 0.0877 - val_loss: 4.6424
Epoch 8/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 2s 101ms/step - accuracy: 0.0972 - loss: 3.9598 - val_accuracy: 0.0

## Make Text Generator Model for prediction. For genearting text, we need to pass one sample at a time.We use same layers which we used early. If we create a new layers then wieghts will be initialized randomly. so we have to use exisitng layers with the trained weights

In [18]:
input2 = Input(shape=(1,)) # Only input one word at a time
x = embedding_layer(input2)
x, h, c = lstm(x, [h_i, c_i]) # now we need states. LSTM needs three inputs. The current input
                                                        # the previous cell and previous hidden state.here x is a single
                                                        # word index
output2 = dense(x)
TextGen_model = Model([input2, h_i, c_i], [output2, h, c])
# h_i, c_i are initial hidden and cell state and  h, c are the next hidden and cell state.

# idx2word dictionary to get back words for sentences during prediction

In [19]:
idx2word = {v:k for k, v in word2idx.items()}

# Write a function to generate one line at a time

In [20]:
def generate_line():
  np_input = np.array([[ word2idx['<start>'] ]]) # The first input word is our input token
  h = np.zeros((1, hidden_size)) # h and c are intially zero which is consistent with our training
  c = np.zeros((1, hidden_size))

  # so we know when to quit
  end = word2idx['<end>']

  # store the output here
  output_sentence = []


  for ii in range(seq_len):
    out, h, c = TextGen_model.predict([np_input, h, c], verbose = 'False')
    # o is the list or word probabilities for the next word and from where we are going to take a sample.
    # h and c are next hidden and cell states

    probs = out[0,0] # sample the first word.
    probs[0] = 0 # set the probabilities to zero if the first word is at zero index
    probs /= probs.sum() # normalize to make it valid prob distribution
    idx = np.random.choice(len(probs), p=probs) # sample the next word
    if idx == end: # if index is last word then break the loop
      break

    # Accumulate output. use word2idx mapping to append the word in our sentence
    output_sentence.append(idx2word.get(idx, 0 % idx))

    # make the next input into model
    np_input[0,0] = idx # make sure that np_input has the latest word

  return ' '.join(output_sentence)


# Generate Five lines of Text

In [21]:
for jj in range(5):
    print(generate_line())

i arthur please home of should mustn't words
there someone in the mica interrupted finger run.
it supposed what the room be rested outsiders. if
then on i quarry talk,
i chose, the papered trouble. lupine cousins:
